# 📄 Salama Insurance — Claim Document Generator for AI Extraction

**Purpose:** Generate realistic insurance claim PDF documents from actual claims data for:
- 🧠 **AI Document Extraction** — Test OCR and document intelligence pipelines (Azure AI Document Intelligence, `ai_parse_document`)
- 📝 **Claims Processing Automation** — Train IDP (Intelligent Document Processing) models
- 🔍 **Entity Recognition** — Validate extraction of claim IDs, amounts, dates, policy numbers
- 📦 **RAG Pipelines** — Build knowledge bases from structured claim documents

**Document Types Generated:**
1. **Claim Submission Form** — Initial claim filing with policyholder details
2. **Settlement Notification** — Approved claims with payment details
3. **Investigation Report** — Fraud investigation findings
4. **Denial Letter** — Rejected claims with reasons

**Data Source:** `salama_insurance.salama_silver.fact_claim` joined with `fact_fraud_investigation`  
**Output:** PDF files + JSON metadata in `/Volumes/salama_insurance/salama_silver/claim_documents/`

In [0]:
%pip install fpdf2 --quiet

In [0]:
import os
import json
import random
import math
from datetime import datetime, timedelta
from pathlib import Path

from fpdf import FPDF

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
OUTPUT_DIR = "/tmp/claim_documents"
VOLUME_PATH = "/Volumes/salama_insurance/salama_silver/claim_documents"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"✅ Output directory: {OUTPUT_DIR}")
print(f"📅 Timestamp: {TIMESTAMP}")

In [0]:
%sql
SELECT
    c.CLAIM_ID,
    c.CLAIM_NUMBER,
    c.POLICY_ID,
    c.BUSINESS_LINE,
    c.CLAIM_TYPE,
    c.CLAIM_STATUS,
    ROUND(c.CLAIMED_AMOUNT, 2)   AS CLAIMED_AMOUNT,
    ROUND(c.APPROVED_AMOUNT, 2)  AS APPROVED_AMOUNT,
    ROUND(c.PAID_AMOUNT, 2)      AS PAID_AMOUNT,
    ROUND(c.RESERVE_AMOUNT, 2)   AS RESERVE_AMOUNT,
    c.RISK_RATING,
    CAST(c.DAYS_TO_SETTLE AS INT)  AS DAYS_TO_SETTLE,
    CAST(c.DAYS_TO_REPORT AS INT)  AS DAYS_TO_REPORT,
    c.ADJUSTER_ID,
    c.FINDINGS,
    c.CL_DATE,
    c.CLAIM_AGING_BUCKET,
    f.INVESTIGATION_ID,
    f.FRAUD_SCORE,
    ROUND(f.INVESTIGATION_COST, 2)      AS INVESTIGATION_COST,
    ROUND(f.FRAUD_AMOUNT_DETECTED, 2)   AS FRAUD_AMOUNT_DETECTED,
    ROUND(f.RECOVERY_AMOUNT, 2)         AS INV_RECOVERY_AMOUNT,
    CAST(f.INVESTIGATION_DAYS AS INT)   AS INV_DAYS,
    f.INVESTIGATOR_ID,
    f.INVESTIGATION_STATUS AS INV_STATUS,
    f.FINDINGS AS INV_FINDINGS
FROM salama_insurance.salama_silver.fact_claim c
LEFT JOIN salama_insurance.salama_silver.fact_fraud_investigation f
    ON c.FACT_FRAUD_KEY = f.FRAUD_KEY_NEW
WHERE c.CLAIM_ID IS NOT NULL
ORDER BY RAND()
LIMIT 15

In [0]:
def _safe(val, default=0):
    """Convert NaN/None to a default."""
    if val is None:
        return default
    if isinstance(val, float) and math.isnan(val):
        return default
    return val


def _sanitize(text):
    """Replace non-latin1 characters for Helvetica font compatibility."""
    replacements = {
        '\u2014': '-', '\u2013': '-',  # em dash, en dash
        '\u2018': "'", '\u2019': "'",  # smart quotes
        '\u201c': '"', '\u201d': '"',  # smart double quotes
        '\u2026': '...', '\u00a0': ' ', '\u2011': '-',
    }
    text = str(text)
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text.encode('latin-1', errors='replace').decode('latin-1')


class ClaimDocumentPDF(FPDF):
    """Custom PDF class for Salama Insurance claim documents."""

    BLUE = (31, 78, 121)
    LIGHT_BLUE = (219, 234, 254)
    DARK = (30, 41, 59)
    GRAY = (100, 116, 139)
    RED = (220, 38, 38)
    GREEN = (22, 163, 74)
    WHITE = (255, 255, 255)

    def header(self):
        self.set_font('Helvetica', 'B', 18)
        self.set_text_color(*self.BLUE)
        self.cell(0, 10, 'SALAMA INSURANCE', ln=True)
        self.set_font('Helvetica', '', 8)
        self.set_text_color(*self.GRAY)
        self.cell(0, 4, 'P.O. Box 12345, Abu Dhabi, UAE  |  Tel: +971-2-555-0100  |  claims@salama-insurance.ae', ln=True)
        self.set_draw_color(*self.BLUE)
        self.set_line_width(0.8)
        self.line(10, self.get_y() + 2, 200, self.get_y() + 2)
        self.ln(6)

    def footer(self):
        self.set_y(-20)
        self.set_draw_color(*self.BLUE)
        self.set_line_width(0.4)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(2)
        self.set_font('Helvetica', 'I', 7)
        self.set_text_color(*self.GRAY)
        self.cell(0, 4, 'Salama Insurance Co. | Licensed by the Insurance Authority of the UAE | CR No. 12345', align='C', ln=True)
        self.cell(0, 4, f'Page {self.page_no()}/{{nb}}  |  This is a computer-generated document. No signature required.', align='C')

    def section_title(self, title):
        self.set_fill_color(*self.BLUE)
        self.set_text_color(*self.WHITE)
        self.set_font('Helvetica', 'B', 11)
        self.cell(0, 8, f'  {title}', fill=True, ln=True)
        self.ln(2)

    def add_field(self, label, value, bold_value=False):
        self.set_font('Helvetica', '', 9)
        self.set_text_color(*self.GRAY)
        self.cell(55, 6, _sanitize(label) + ':', align='R')
        self.set_text_color(*self.DARK)
        self.set_font('Helvetica', 'B' if bold_value else '', 9)
        self.cell(0, 6, f'  {_sanitize(value)}', ln=True)

    def add_field_row(self, fields):
        """Add multiple label-value pairs in a single row."""
        w = 190 // len(fields)
        for label, value in fields:
            self.set_font('Helvetica', '', 8)
            self.set_text_color(*self.GRAY)
            self.cell(w // 2, 6, label + ':')
            self.set_text_color(*self.DARK)
            self.set_font('Helvetica', 'B', 9)
            self.cell(w // 2, 6, str(value))
        self.ln()

    def add_amount_table(self, rows):
        """Add a financial summary table."""
        self.set_fill_color(*self.LIGHT_BLUE)
        self.set_font('Helvetica', 'B', 9)
        self.set_text_color(*self.BLUE)
        self.cell(95, 7, '  Description', fill=True, border=1)
        self.cell(50, 7, '  Amount (AED)', fill=True, border=1, ln=True)
        self.set_font('Helvetica', '', 9)
        self.set_text_color(*self.DARK)
        for desc, amt in rows:
            self.cell(95, 7, f'  {_sanitize(desc)}', border=1)
            self.cell(50, 7, f'  {_sanitize(amt)}', border=1, ln=True)
        self.ln(2)

    def stamp_box(self, text, color):
        """Add a status stamp box."""
        self.set_fill_color(*color)
        self.set_text_color(*self.WHITE)
        self.set_font('Helvetica', 'B', 14)
        x = 140
        self.set_xy(x, 30)
        self.cell(60, 12, text, fill=True, align='C', border=1)
        self.set_xy(10, self.get_y() + 14)


print("✅ ClaimDocumentPDF class defined with header, footer, sections, tables, stamps")

In [0]:
# --- Fake customer name / address generators ---
_first = ["Ahmad", "Fatima", "Omar", "Layla", "Khalid", "Nora", "Yusuf", "Amira", "Hassan", "Mariam"]
_last  = ["Al-Rashid", "Hassan", "Malik", "Noor", "Ibrahim", "Saleh", "Khan", "Farouk", "Zayed", "Mansouri"]
_city  = ["Abu Dhabi", "Dubai", "Sharjah", "Al Ain", "Ajman", "Ras Al Khaimah", "Fujairah"]

def _fake_customer():
    return {
        "name": f"{random.choice(_first)} {random.choice(_last)}",
        "id": f"CUS-{random.randint(10000,99999)}",
        "phone": f"+971-{random.randint(50,56)}-{random.randint(100,999)}-{random.randint(1000,9999)}",
        "email": f"{random.choice(_first).lower()}.{random.choice(_last).lower()}@email.com",
        "address": f"{random.randint(1,999)} {random.choice(['Al Falah','Corniche','Khalifa','Hamdan','Electra'])} St, {random.choice(_city)}, UAE",
    }

def _fmt_date(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return "N/A"
    try:
        return val.strftime("%d %B %Y")
    except:
        return str(val)[:10]

def _fmt_amt(val):
    v = _safe(val, 0)
    return f"AED {float(v):,.2f}"


# =====================================================================
# TEMPLATE 1: Claim Submission Form
# =====================================================================
def generate_claim_form(claim, output_dir):
    cust = _fake_customer()
    pdf = ClaimDocumentPDF()
    pdf.alias_nb_pages()
    pdf.add_page()
    pdf.stamp_box("CLAIM FORM", ClaimDocumentPDF.BLUE)

    # Document title
    pdf.set_font('Helvetica', 'B', 14)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    pdf.cell(0, 10, 'Insurance Claim Submission Form', ln=True)
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(*ClaimDocumentPDF.GRAY)
    pdf.cell(0, 5, _sanitize(f'Reference: {claim["CLAIM_NUMBER"]}  |  Date: {_fmt_date(claim.get("CL_DATE"))}'), ln=True)
    pdf.ln(4)

    # Policyholder
    pdf.section_title('POLICYHOLDER INFORMATION')
    pdf.add_field('Full Name', cust['name'], bold_value=True)
    pdf.add_field('Customer ID', cust['id'])
    pdf.add_field('Contact Phone', cust['phone'])
    pdf.add_field('Email Address', cust['email'])
    pdf.add_field('Mailing Address', cust['address'])
    pdf.ln(3)

    # Policy details
    pdf.section_title('POLICY DETAILS')
    pdf.add_field('Policy Number', str(claim['POLICY_ID']), bold_value=True)
    pdf.add_field('Business Line', str(_safe(claim.get('BUSINESS_LINE'), 'N/A')))
    pdf.add_field('Claim Type', str(_safe(claim.get('CLAIM_TYPE'), 'N/A')).replace('_', ' ').title())
    pdf.add_field('Risk Rating', str(_safe(claim.get('RISK_RATING'), 'N/A')))
    pdf.ln(3)

    # Claim details
    pdf.section_title('CLAIM DETAILS')
    pdf.add_field('Claim Number', str(claim['CLAIM_NUMBER']), bold_value=True)
    pdf.add_field('Claim ID', str(claim['CLAIM_ID']))
    pdf.add_field('Date of Incident', _fmt_date(claim.get('CL_DATE')))
    pdf.add_field('Days to Report', str(_safe(claim.get('DAYS_TO_REPORT'), 'N/A')))
    pdf.add_field('Current Status', str(_safe(claim.get('CLAIM_STATUS'), 'N/A')))
    pdf.ln(3)

    # Financial
    pdf.section_title('FINANCIAL SUMMARY')
    pdf.add_amount_table([
        ('Claimed Amount', _fmt_amt(claim.get('CLAIMED_AMOUNT'))),
        ('Approved Amount', _fmt_amt(claim.get('APPROVED_AMOUNT'))),
        ('Paid Amount', _fmt_amt(claim.get('PAID_AMOUNT'))),
        ('Reserve Amount', _fmt_amt(claim.get('RESERVE_AMOUNT'))),
    ])

    # Declaration
    pdf.ln(4)
    pdf.section_title('DECLARATION')
    pdf.set_font('Helvetica', '', 8)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    pdf.multi_cell(0, 4, _sanitize(
        'I hereby declare that the information provided in this claim form is true and accurate '
        'to the best of my knowledge. I understand that any false or misleading information '
        'may result in the denial of this claim and potential legal action. I authorize Salama '
        'Insurance to verify any information provided and to obtain necessary records.'))
    pdf.ln(6)
    pdf.add_field('Policyholder Signature', f'_________________  ({cust["name"]})')
    pdf.add_field('Date', _fmt_date(datetime.now()))

    safe_id = str(claim['CLAIM_ID']).replace('/', '_')
    path = os.path.join(output_dir, f"claim_form_{safe_id}.pdf")
    pdf.output(path)
    return path, "Claim Submission Form", cust


# =====================================================================
# TEMPLATE 2: Settlement Notification Letter
# =====================================================================
def generate_settlement_letter(claim, output_dir):
    cust = _fake_customer()
    pdf = ClaimDocumentPDF()
    pdf.alias_nb_pages()
    pdf.add_page()
    pdf.stamp_box("SETTLED", ClaimDocumentPDF.GREEN)

    pdf.set_font('Helvetica', 'B', 14)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    pdf.cell(0, 10, 'Claim Settlement Notification', ln=True)
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(*ClaimDocumentPDF.GRAY)
    pdf.cell(0, 5, _sanitize(f'Date: {_fmt_date(datetime.now())}  |  Ref: {claim["CLAIM_NUMBER"]}/SETTLE'), ln=True)
    pdf.ln(4)

    pdf.set_font('Helvetica', '', 10)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    pdf.multi_cell(0, 5, _sanitize(
        f'Dear {cust["name"]},\n\n'
        f'We are pleased to inform you that your insurance claim {claim["CLAIM_NUMBER"]} '
        f'has been reviewed and approved for settlement. Please find the details below.'))
    pdf.ln(4)

    pdf.section_title('CLAIM REFERENCE')
    pdf.add_field('Claim Number', str(claim['CLAIM_NUMBER']), bold_value=True)
    pdf.add_field('Policy Number', str(claim['POLICY_ID']))
    pdf.add_field('Business Line', str(_safe(claim.get('BUSINESS_LINE'), 'N/A')))
    pdf.add_field('Claim Type', str(_safe(claim.get('CLAIM_TYPE'), 'N/A')).replace('_', ' ').title())
    pdf.add_field('Processing Duration', f"{_safe(claim.get('DAYS_TO_SETTLE'), 'N/A')} days")
    pdf.ln(3)

    pdf.section_title('SETTLEMENT DETAILS')
    pdf.add_amount_table([
        ('Original Claimed Amount', _fmt_amt(claim.get('CLAIMED_AMOUNT'))),
        ('Approved Settlement Amount', _fmt_amt(claim.get('APPROVED_AMOUNT'))),
        ('Amount Disbursed', _fmt_amt(claim.get('PAID_AMOUNT'))),
    ])

    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    pdf.multi_cell(0, 5, _sanitize(
        f'The settlement amount of {_fmt_amt(claim.get("PAID_AMOUNT"))} has been '
        f'transferred to your registered bank account. Please allow 3-5 business days '
        f'for the funds to reflect.\n\n'
        f'If you have any questions regarding this settlement, please contact our '
        f'Claims Department at claims@salama-insurance.ae or call +971-2-555-0100.'))
    pdf.ln(6)
    pdf.add_field('Authorized By', f'Claims Department — Salama Insurance')
    pdf.add_field('Adjuster Reference', str(_safe(claim.get('ADJUSTER_ID'), 'N/A')))

    safe_id = str(claim['CLAIM_ID']).replace('/', '_')
    path = os.path.join(output_dir, f"settlement_{safe_id}.pdf")
    pdf.output(path)
    return path, "Settlement Notification", cust


# =====================================================================
# TEMPLATE 3: Investigation Report
# =====================================================================
def generate_investigation_report(claim, output_dir):
    cust = _fake_customer()
    pdf = ClaimDocumentPDF()
    pdf.alias_nb_pages()
    pdf.add_page()

    inv_findings = str(_safe(claim.get('INV_FINDINGS'), 'INCONCLUSIVE'))
    color = ClaimDocumentPDF.RED if 'FRAUD' in inv_findings.upper() else ClaimDocumentPDF.BLUE
    pdf.stamp_box('CONFIDENTIAL', color)

    pdf.set_font('Helvetica', 'B', 14)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    pdf.cell(0, 10, 'Fraud Investigation Report', ln=True)
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(*ClaimDocumentPDF.GRAY)
    pdf.cell(0, 5, _sanitize(f'Investigation Ref: {_safe(claim.get("INVESTIGATION_ID"), "N/A")}  |  Classification: RESTRICTED'), ln=True)
    pdf.ln(4)

    pdf.section_title('CLAIM UNDER INVESTIGATION')
    pdf.add_field('Claim Number', str(claim['CLAIM_NUMBER']), bold_value=True)
    pdf.add_field('Policy Number', str(claim['POLICY_ID']))
    pdf.add_field('Business Line', str(_safe(claim.get('BUSINESS_LINE'), 'N/A')))
    pdf.add_field('Claim Type', str(_safe(claim.get('CLAIM_TYPE'), 'N/A')).replace('_', ' ').title())
    pdf.add_field('Claimed Amount', _fmt_amt(claim.get('CLAIMED_AMOUNT')))
    pdf.add_field('Claimant', cust['name'])
    pdf.ln(3)

    pdf.section_title('INVESTIGATION DETAILS')
    pdf.add_field('Investigation ID', str(_safe(claim.get('INVESTIGATION_ID'), 'N/A')))
    pdf.add_field('Investigator', str(_safe(claim.get('INVESTIGATOR_ID'), 'N/A')))
    pdf.add_field('Investigation Status', str(_safe(claim.get('INV_STATUS'), 'N/A')))
    pdf.add_field('Investigation Duration', f"{_safe(claim.get('INV_DAYS'), 'N/A')} days")
    fraud_score = _safe(claim.get('FRAUD_SCORE'), 0)
    pdf.add_field('Fraud Risk Score', f"{float(fraud_score):.1f} / 100")
    pdf.ln(3)

    pdf.section_title('FINANCIAL IMPACT')
    pdf.add_amount_table([
        ('Claimed Amount', _fmt_amt(claim.get('CLAIMED_AMOUNT'))),
        ('Fraud Amount Detected', _fmt_amt(claim.get('FRAUD_AMOUNT_DETECTED'))),
        ('Recovery Amount', _fmt_amt(claim.get('INV_RECOVERY_AMOUNT'))),
        ('Investigation Cost', _fmt_amt(claim.get('INVESTIGATION_COST'))),
    ])

    pdf.section_title('FINDINGS & CONCLUSION')
    findings_text = inv_findings.replace('_', ' ').title()
    pdf.set_font('Helvetica', 'B', 11)
    pdf.set_text_color(*ClaimDocumentPDF.RED if 'FRAUD' in inv_findings.upper() else ClaimDocumentPDF.GREEN)
    pdf.cell(0, 8, _sanitize(f'Finding: {findings_text}'), ln=True)
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)

    narratives = {
        'FRAUD_CONFIRMED': 'Based on documentary evidence, witness statements, and financial analysis, '
                           'this investigation has confirmed fraudulent activity. The claim contained '
                           'material misrepresentations regarding the reported incident. Evidence suggests '
                           'intentional inflation of claimed damages.',
        'FRAUD_SUSPECTED': 'Multiple indicators of potential fraud were identified during the investigation. '
                           'While definitive proof was not obtained, the pattern of inconsistencies warrants '
                           'continued monitoring and potential referral to law enforcement.',
        'INCONCLUSIVE':    'The investigation did not yield sufficient evidence to confirm or deny '
                           'fraudulent activity. The claim will proceed through standard processing '
                           'channels with enhanced monitoring.',
        'NO_FRAUD':        'After thorough investigation, no evidence of fraud was found. The claim '
                           'appears to be legitimate and is recommended for standard processing.',
    }
    narrative = narratives.get(inv_findings, narratives['INCONCLUSIVE'])
    pdf.multi_cell(0, 5, _sanitize(f'\n{narrative}'))
    pdf.ln(3)
    pdf.add_field('Lead Investigator', str(_safe(claim.get('INVESTIGATOR_ID'), 'N/A')))
    pdf.add_field('Report Date', _fmt_date(datetime.now()))
    pdf.add_field('Distribution', 'Claims Director, Legal Department, Compliance')

    safe_id = str(claim['CLAIM_ID']).replace('/', '_')
    path = os.path.join(output_dir, f"investigation_{safe_id}.pdf")
    pdf.output(path)
    return path, "Investigation Report", cust


# =====================================================================
# TEMPLATE 4: Denial Letter
# =====================================================================
def generate_denial_letter(claim, output_dir):
    cust = _fake_customer()
    pdf = ClaimDocumentPDF()
    pdf.alias_nb_pages()
    pdf.add_page()
    pdf.stamp_box('REJECTED', ClaimDocumentPDF.RED)

    pdf.set_font('Helvetica', 'B', 14)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    pdf.cell(0, 10, 'Claim Denial Notification', ln=True)
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(*ClaimDocumentPDF.GRAY)
    pdf.cell(0, 5, _sanitize(f'Date: {_fmt_date(datetime.now())}  |  Ref: {claim["CLAIM_NUMBER"]}/DENY'), ln=True)
    pdf.ln(4)

    pdf.set_font('Helvetica', '', 10)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    pdf.multi_cell(0, 5, _sanitize(
        f'Dear {cust["name"]},\n\n'
        f'We regret to inform you that after careful review, your insurance claim '
        f'{claim["CLAIM_NUMBER"]} has been denied. This letter outlines the details '
        f'of our decision and your options for appeal.'))
    pdf.ln(4)

    pdf.section_title('CLAIM DETAILS')
    pdf.add_field('Claim Number', str(claim['CLAIM_NUMBER']), bold_value=True)
    pdf.add_field('Policy Number', str(claim['POLICY_ID']))
    pdf.add_field('Business Line', str(_safe(claim.get('BUSINESS_LINE'), 'N/A')))
    pdf.add_field('Claim Type', str(_safe(claim.get('CLAIM_TYPE'), 'N/A')).replace('_', ' ').title())
    pdf.add_field('Claimed Amount', _fmt_amt(claim.get('CLAIMED_AMOUNT')))
    pdf.add_field('Risk Assessment', str(_safe(claim.get('RISK_RATING'), 'N/A')))
    pdf.ln(3)

    pdf.section_title('REASON FOR DENIAL')
    findings = str(_safe(claim.get('FINDINGS'), '')).replace('_', ' ').lower()
    reasons = [
        'The claim does not fall within the covered perils of your policy.',
        f'Investigation findings: {findings}.' if findings and findings != 'none' else 'Insufficient supporting documentation provided.',
        'The incident described does not meet the threshold requirements for the claim type filed.',
    ]
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(*ClaimDocumentPDF.DARK)
    for i, reason in enumerate(reasons, 1):
        pdf.cell(0, 6, _sanitize(f'  {i}. {reason}'), ln=True)
    pdf.ln(3)

    pdf.section_title('APPEAL PROCESS')
    pdf.set_font('Helvetica', '', 9)
    pdf.multi_cell(0, 5, _sanitize(
        'You have the right to appeal this decision within 30 calendar days of receiving '
        'this notification. To file an appeal:\n\n'
        '  1. Submit a written appeal letter stating your grounds for reconsideration\n'
        '  2. Provide any additional supporting documentation\n'
        '  3. Send to: Claims Appeals Committee, Salama Insurance, P.O. Box 12345, Abu Dhabi\n'
        '  4. Or email: appeals@salama-insurance.ae\n\n'
        'Your appeal will be reviewed by our Senior Claims Committee within 15 business days.'))

    pdf.ln(6)
    pdf.add_field('Claims Department', 'Salama Insurance Co.')
    pdf.add_field('Adjuster Reference', str(_safe(claim.get('ADJUSTER_ID'), 'N/A')))

    safe_id = str(claim['CLAIM_ID']).replace('/', '_')
    path = os.path.join(output_dir, f"denial_{safe_id}.pdf")
    pdf.output(path)
    return path, "Denial Letter", cust


print("✅ 4 document templates defined:")
print("   1. Claim Submission Form")
print("   2. Settlement Notification")
print("   3. Investigation Report")
print("   4. Denial Letter")

In [0]:
import pandas as pd

df_claims = _sqldf.toPandas()
print(f"📎 Loaded {len(df_claims)} claims for document generation\n")

# --- Map claim status to document type ---
def pick_template(row):
    status = str(_safe(row.get('CLAIM_STATUS'), '')).upper()
    has_investigation = _safe(row.get('INVESTIGATION_ID'), None) is not None

    if status in ('APPROVED', 'SETTLED'):
        return 'settlement'
    elif status == 'REJECTED':
        return 'denial'
    elif has_investigation:
        return 'investigation'
    else:
        return 'claim_form'

generators = {
    'claim_form':     generate_claim_form,
    'settlement':     generate_settlement_letter,
    'investigation':  generate_investigation_report,
    'denial':         generate_denial_letter,
}

# --- Generate documents ---
results = []
all_metadata = []

for idx, row in df_claims.iterrows():
    claim_id = str(row['CLAIM_ID'])
    status = str(row['CLAIM_STATUS'])
    doc_type = pick_template(row.to_dict())
    gen_fn = generators[doc_type]

    print(f"\n📄 [{idx+1}/{len(df_claims)}] {claim_id} (Status: {status}) → {doc_type}")

    try:
        path, doc_name, cust = gen_fn(row.to_dict(), OUTPUT_DIR)
        size_kb = round(os.path.getsize(path) / 1024, 1)
        print(f"   ✅ {os.path.basename(path)} ({size_kb} KB)")

        meta = {
            "document_id": f"DOC-{claim_id}-{TIMESTAMP}",
            "claim_id": claim_id,
            "claim_number": str(row['CLAIM_NUMBER']),
            "policy_id": str(row['POLICY_ID']),
            "document_type": doc_name,
            "template": doc_type,
            "claim_status": status,
            "business_line": str(_safe(row.get('BUSINESS_LINE'), 'N/A')),
            "claim_type": str(_safe(row.get('CLAIM_TYPE'), 'N/A')),
            "claimed_amount": float(_safe(row.get('CLAIMED_AMOUNT'), 0)),
            "approved_amount": float(_safe(row.get('APPROVED_AMOUNT'), 0)),
            "customer_name": cust['name'],
            "customer_id": cust['id'],
            "file_name": os.path.basename(path),
            "file_size_kb": size_kb,
            "generated_at": datetime.now().isoformat(),
        }

        results.append({**meta, "file_path": path})
        all_metadata.append(meta)

    except Exception as e:
        print(f"   ❌ Error: {e}")

print(f"\n{'='*60}")
print(f"✅ Generated {len(results)} / {len(df_claims)} documents")
print(f"{'='*60}")

df_results = pd.DataFrame(results)
display(df_results[['claim_id', 'claim_number', 'document_type', 'claim_status', 'business_line', 'file_size_kb', 'file_name']])

In [0]:
import shutil

# Save combined metadata JSON
meta_path = os.path.join(OUTPUT_DIR, "document_metadata.json")
with open(meta_path, 'w') as f:
    json.dump({
        "generated_at": datetime.now().isoformat(),
        "total_documents": len(all_metadata),
        "source_tables": [
            "salama_insurance.salama_silver.fact_claim",
            "salama_insurance.salama_silver.fact_fraud_investigation"
        ],
        "document_types": list(set(m['document_type'] for m in all_metadata)),
        "documents": all_metadata,
    }, f, indent=2)
print(f"✅ Metadata saved: {meta_path}")

# Create the UC Volume if needed, then copy files
spark.sql(f"CREATE VOLUME IF NOT EXISTS salama_insurance.salama_silver.claim_documents")
print(f"\u2705 Volume ready: {VOLUME_PATH}")

copied = 0
total_kb = 0
for fname in sorted(os.listdir(OUTPUT_DIR)):
    src = os.path.join(OUTPUT_DIR, fname)
    dst = os.path.join(VOLUME_PATH, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        size_kb = round(os.path.getsize(dst) / 1024, 1)
        total_kb += size_kb
        copied += 1
        print(f"   ✅ {fname:50s} {size_kb:>8.1f} KB")

print(f"\n{'='*70}")
print(f"✅ Copied {copied} files to {VOLUME_PATH}")
print(f"   Total size: {total_kb:.1f} KB ({total_kb/1024:.1f} MB)")

In [0]:
import pandas as pd

print("\n" + "="*70)
print("📊 DOCUMENT GENERATION SUMMARY")
print("="*70)

# By document type
print("\n📁 By Document Type:")
display(df_results.groupby('document_type').agg(
    count=('claim_id', 'count'),
    avg_size_kb=('file_size_kb', 'mean'),
    total_size_kb=('file_size_kb', 'sum')
).reset_index())

# By business line
print("\n🏢 By Business Line:")
display(df_results.groupby('business_line').agg(
    count=('claim_id', 'count'),
    doc_types=('document_type', lambda x: ', '.join(sorted(set(x))))
).reset_index())

# List all files in volume
print(f"\n📂 Files in {VOLUME_PATH}:")
for f in sorted(os.listdir(VOLUME_PATH)):
    fp = os.path.join(VOLUME_PATH, f)
    print(f"   {f:50s} {round(os.path.getsize(fp)/1024, 1):>8.1f} KB")

print(f"\n" + "="*70)
print("🚀 NEXT STEPS: AI DOCUMENT EXTRACTION")
print("="*70)
print("""
1. AZURE DOCUMENT INTELLIGENCE:
   Use the pre-built or custom models to extract entities from the PDFs.

2. DATABRICKS ai_parse_document():
   SELECT ai_parse_document(
       content,
       'claim_number STRING, policy_number STRING, claimed_amount DOUBLE'
   ) FROM read_files('/Volumes/salama_insurance/salama_silver/claim_documents/*.pdf')

3. VALIDATE EXTRACTION:
   Compare extracted values against document_metadata.json ground truth.

4. RAG PIPELINE:
   Chunk the PDFs and index them for retrieval-augmented generation.
""")

## ℹ️ About These Documents

**These are synthetic claim documents** generated from real claims data in `salama_insurance.salama_silver.fact_claim` joined with `salama_insurance.salama_silver.fact_fraud_investigation`.

### Document Types
| Type | Trigger | Contents |
|------|---------|----------|
| **Claim Form** | REPORTED / default | Policyholder info, policy details, claim details, financial summary, declaration |
| **Settlement Letter** | APPROVED / SETTLED | Settlement amount, payment details, processing duration |
| **Investigation Report** | Has linked fraud investigation | Fraud score, investigation findings, financial impact, narrative |
| **Denial Letter** | REJECTED | Denial reasons, appeal process, supporting documentation requirements |

### Key Extractable Entities
- Claim Number, Policy Number, Claim ID
- Customer Name, Address, Phone, Email
- Claimed Amount, Approved Amount, Paid Amount
- Business Line, Claim Type, Risk Rating
- Investigation ID, Fraud Score, Findings
- Dates, Status codes, Adjuster references

---
*Generated by Salama Insurance Claims Document Generator for AI Extraction*